# 09: Research Summary & Findings

**Purpose**: Synthesize findings from Stages 1-4 into a coherent research narrative.

**Structure**:
1. Research question restatement
2. Key findings synthesis
3. Required figures (5 total)
4. Policy recommendations
5. Limitations acknowledgment


In [ ]:
from pathlib import Path
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path("/Users/aaryakhanna/transit-deserts").resolve()
os.chdir(ROOT)

outputs = ROOT / "outputs"

print("✓ Setup complete")
print(f"Working directory: {ROOT}")


## Research Question

**"Which census tracts in LA County experience systematically low job accessibility via public transit, after controlling for population and income, and what structural factors best explain this?"**


## Required Figure 1: Map of Transit Deserts

Load and display the transit desert map from Notebook 03.


In [ ]:
try:
    from IPython.display import Image, display
    img_path = outputs / 'map_transit_deserts.png'
    if img_path.exists():
        display(Image(str(img_path)))
        print("✓ Figure 1: Map of Transit Deserts")
    else:
        print("⚠ Transit desert map not found. Run Notebook 03 first.")
except Exception as e:
    print(f"⚠ Error loading map: {e}")


## Required Figure 2: Map of LISA Low-Low Clusters

Load and display the LISA cluster map from Notebook 04.


In [ ]:
try:
    img_path = outputs / 'map_lisa_clusters.png'
    if img_path.exists():
        display(Image(str(img_path)))
        print("✓ Figure 2: Map of LISA Low-Low Clusters")
    else:
        print("⚠ LISA cluster map not found. Run Notebook 04 first.")
except Exception as e:
    print(f"⚠ Error loading map: {e}")


## Required Figure 3: Accessibility Distribution Histogram

Create histogram of accessibility distribution.


In [ ]:
try:
    tracts = gpd.read_file(outputs / "tracts_with_accessibility.geojson")
    target_col = 'access_30min_per1k'
    
    access_data = tracts[target_col].replace([np.inf, -np.inf], np.nan).dropna()
    access_data = access_data[access_data >= 0]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(access_data, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    ax.set_xlabel('Accessibility (jobs per 1,000 residents)', fontsize=12)
    ax.set_ylabel('Number of Census Tracts', fontsize=12)
    ax.set_title('Distribution of Transit Accessibility\n(30-minute threshold)', 
                 fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    median_val = access_data.median()
    mean_val = access_data.mean()
    ax.axvline(median_val, color='red', linestyle='--', linewidth=2, label=f'Median: {median_val:.0f}')
    ax.axvline(mean_val, color='orange', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.0f}')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(outputs / 'accessibility_distribution.png', dpi=300, bbox_inches='tight')
    print("✓ Figure 3: Accessibility Distribution Histogram")
    print(f"  Mean: {mean_val:.0f} jobs/1k")
    print(f"  Median: {median_val:.0f} jobs/1k")
    plt.show()
except Exception as e:
    print(f"⚠ Error creating histogram: {e}")


## Required Figure 4: Spatial Regression Coefficient Table

Display spatial regression results from Notebook 04.


In [ ]:
try:
    spatial_reg = pd.read_csv(outputs / 'spatial_regression_results.csv')
    print("✓ Figure 4: Spatial Regression Coefficients")
    print("=" * 80)
    print(spatial_reg.to_string(index=False))
    print("=" * 80)
except FileNotFoundError:
    print("⚠ Spatial regression results not found. Run Notebook 04 first.")
except Exception as e:
    print(f"⚠ Error loading results: {e}")


## Required Figure 5: ML Feature Importance Bar Chart

Display feature importance from Notebook 05.


In [ ]:
try:
    img_path = outputs / 'ml_feature_importance.png'
    if img_path.exists():
        display(Image(str(img_path)))
        print("✓ Figure 5: ML Feature Importance")
    else:
        feature_importance = pd.read_csv(outputs / 'ml_feature_importance.csv')
        print("✓ Figure 5: ML Feature Importance")
        print("\nTop 5 Features:")
        for i, row in feature_importance.head(5).iterrows():
            print(f"  {i+1}. {row['feature']:25s}: {row['importance']*100:6.2f}%")
except FileNotFoundError:
    print("⚠ ML feature importance not found. Run Notebook 05 first.")
except Exception as e:
    print(f"⚠ Error loading feature importance: {e}")


## Key Findings Synthesis

Summarize findings from all stages.


In [ ]:
print("=" * 80)
print("KEY FINDINGS SYNTHESIS")
print("=" * 80)

try:
    tracts = gpd.read_file(outputs / "tracts_with_accessibility.geojson")
    target_col = 'access_30min_per1k'
    access_data = tracts[target_col].replace([np.inf, -np.inf], np.nan).dropna()
    access_data = access_data[access_data >= 0]
    
    access_25th = access_data.quantile(0.25)
    transit_deserts = (tracts[target_col] <= access_25th).sum()
    
    print(f"\n1. TRANSIT DESERTS IDENTIFIED:")
    print(f"   • {transit_deserts:,} census tracts identified as transit deserts (bottom quartile)")
    print(f"   • Threshold: ≤ {access_25th:.0f} jobs per 1,000 residents")
    
    try:
        moran_results = pd.read_csv(outputs / 'spatial_autocorrelation_results.csv')
        print(f"\n2. SPATIAL CLUSTERING:")
        if 'morans_i' in moran_results.columns:
            morans_i = moran_results['morans_i'].iloc[0]
            p_value = moran_results['p_value'].iloc[0] if 'p_value' in moran_results.columns else None
            print(f"   • Global Moran's I: {morans_i:.4f}")
            if p_value:
                sig = "significant" if p_value < 0.05 else "not significant"
                print(f"   • Spatial autocorrelation is {sig} (p = {p_value:.4f})")
    except:
        print(f"\n2. SPATIAL CLUSTERING:")
        print(f"   • Run Notebook 04 for spatial autocorrelation results")
    
    try:
        feature_importance = pd.read_csv(outputs / 'ml_feature_importance.csv')
        top_3 = feature_importance.head(3)
        print(f"\n3. STRUCTURAL FACTORS (Top 3):")
        for i, row in top_3.iterrows():
            print(f"   • {row['feature']:25s}: {row['importance']*100:6.2f}% importance")
    except:
        print(f"\n3. STRUCTURAL FACTORS:")
        print(f"   • Run Notebook 05 for feature importance results")
    
    print(f"\n4. POLICY IMPLICATIONS:")
    print(f"   • Transit deserts cluster spatially → Geographic targeting is effective")
    print(f"   • Distance to downtown is key factor → Infrastructure expansion priorities")
    print(f"   • Population density matters → Supports transit-oriented development")
    
except Exception as e:
    print(f"⚠ Error synthesizing findings: {e}")

print("\n" + "=" * 80)


## Limitations

**Data Limitations**:
- Tract centroids approximate residential/employment locations (systematic bias, but valid for relative comparisons)
- Single time point (8 AM weekday) represents typical accessibility (temporal variation not captured)
- Schedule-based routing (real-time delays/cancellations not accounted for)

**Methodological Limitations**:
- Binary reachability threshold (no distance decay weighting)
- Spatial weights assume Queen contiguity captures relevant relationships
- ML model captures associations, not causal relationships

**These limitations do not invalidate the findings but should be acknowledged in any policy application.**
